## Методические указания по выполнению практикума №4

[МУ блока](README.md) · [Общие МУ](../../../docs/guidelines-students.md) · [Рубрика оценивания](../teachers-assessment/README.md)

**Тема: Применение алгоритмов и библиотек компьютерного зрения для прикладных задач сегментации: segmentation-models-pytorch и Albumentations**

**Тема РПД:** Л14/П4. **Индикатор:** DL-3.2, уровень С.

**Ноутбук:** `#7 P4_Segmentation_SMP.ipynb`. В ЛР4 (Mask R-CNN) вы получали маски отдельных экземпляров готовой моделью; здесь решается другая задача — семантическая сегментация, и модель обучается вами.

**Цель работы:** построить воспроизводимый конвейер семантической сегментации на библиотеке `segmentation-models-pytorch` с аугментациями `Albumentations`, корректно измерить качество по IoU и Dice и в контролируемой серии определить вклад одного выбранного фактора.

**Задачи:**

- Разобрать архитектуру энкодер-декодер со skip-соединениями и роль предобученного энкодера.
- Подготовить данные Oxford-IIIT Pet с trimap-разметкой, преобразовав её в маски классов.
- Собрать пайплайн аугментаций, синхронно применяемый к изображению и маске.
- Обучить baseline `smp.Unet` с энкодером, предобученным на ImageNet.
- Реализовать комбинированную функцию потерь и оценку по mIoU и Dice.
- Провести контролируемую серию с изменением одного фактора и свести результаты в сопоставимую таблицу.
- Выполнить анализ ошибок по худшим по IoU изображениям.

### 1. Теоретическая часть

#### 1.1 Постановка задачи

Семантическая сегментация — попиксельная классификация. Модель отображает изображение $x \in \mathbb{R}^{H \times W \times 3}$ в карту логитов $\hat{y} \in \mathbb{R}^{H \times W \times C}$, где $C$ — число классов. В отличие от instance-сегментации (ЛР4, Mask R-CNN) экземпляры одного класса не различаются: два кота на изображении дадут одну связную область класса «животное».

Базовая функция потерь — попиксельная кросс-энтропия:

$$\mathcal{L}_{CE} = -\frac{1}{HW} \sum_{i=1}^{HW} \sum_{c=1}^{C} y_{i,c} \log p_{i,c}, \qquad p_{i,c} = \frac{e^{\hat{y}_{i,c}}}{\sum_{k} e^{\hat{y}_{i,k}}}.$$

Она оптимизирует каждый пиксель независимо и потому чувствительна к дисбалансу: если 80 % пикселей — фон, модель, предсказывающая только фон, уже получает низкий loss.

#### 1.2 Архитектуры

**U-Net** ([Ronneberger et al., 2015](https://arxiv.org/abs/1505.04597)) — симметричный энкодер-декодер. Энкодер понижает разрешение и наращивает число каналов, декодер восстанавливает разрешение. Ключевой элемент — **skip-соединения**: карты признаков энкодера конкатенируются с соответствующими картами декодера. Без них декодер восстанавливает границы объектов по сильно сжатому представлению и маска получается «размытой».

**DeepLabV3+** использует атрозные (dilated) свёртки и модуль ASPP, собирающий контекст на нескольких масштабах при одном разрешении. **SegFormer** заменяет свёрточный энкодер иерархическим трансформером с облегчённой MLP-головой.

**Библиотека `segmentation-models-pytorch` (SMP)** разделяет декодер и энкодер: декодер (`Unet`, `UnetPlusPlus`, `FPN`, `DeepLabV3Plus`, ...) комбинируется с любым энкодером из большого списка (`resnet34`, `efficientnet-b0`, `mit_b0`, ...), причём энкодер можно взять предобученным: `encoder_weights="imagenet"`. Это и есть перенос обучения в сегментации — признаки, выученные на классификации, переиспользуются как признаки для разметки пикселей.

Важно: предобученный энкодер требует ровно той нормализации входа, с которой он обучался. Несовпадение нормализации — самая частая причина «необъяснимо низкого» качества.

#### 1.3 Метрики и функции потерь

Для класса $c$ с множеством предсказанных пикселей $A$ и истинных $B$:

$$\mathrm{IoU}(A, B) = \frac{|A \cap B|}{|A \cup B|}, \qquad \mathrm{Dice}(A, B) = \frac{2 |A \cap B|}{|A| + |B|}.$$

Метрики монотонно связаны:

$$\mathrm{Dice} = \frac{2\,\mathrm{IoU}}{1 + \mathrm{IoU}},$$

поэтому ранжирование моделей по ним совпадает, но абсолютные значения различаются — Dice всегда не меньше IoU. Указывать в отчёте, какая именно метрика приведена, обязательно. Итоговая метрика — **mIoU**, среднее IoU по классам; усреднение по классам, а не по пикселям, не даёт крупному фоновому классу маскировать провал на редком классе.

Дифференцируемый аналог Dice используется как функция потерь. На практике применяют сумму:

$$\mathcal{L} = \lambda \,\mathcal{L}_{CE} + (1 - \lambda) \,\mathcal{L}_{Dice},$$

где CE даёт устойчивый градиент на ранних эпохах, а Dice напрямую оптимизирует перекрытие областей и лучше переносит дисбаланс классов. В SMP доступны `smp.losses.DiceLoss`, `smp.losses.FocalLoss`, `smp.losses.TverskyLoss`.

#### 1.4 Данные и аугментации

Oxford-IIIT Pet содержит **trimap**-разметку: каждый пиксель размечен одним из трёх значений — `1` (животное), `2` (фон), `3` (граница/неопределённость). Граничная зона выделена отдельно потому, что в ней разметка ненадёжна. Возможны два корректных решения: (а) считать границу третьим классом; (б) исключить её из подсчёта потерь и метрик через `ignore_index`. Выбор влияет на абсолютные значения метрик, поэтому он фиксируется один раз для всех серий.

Аугментации в сегментации имеют жёсткое ограничение: **любое геометрическое преобразование должно быть применено к изображению и маске одновременно и согласованно**. `Albumentations` решает это конструкцией `transform(image=image, mask=mask)`. Дополнительно:

- маска интерполируется только методом ближайшего соседа — линейная интерполяция породит несуществующие метки классов;
- фотометрические преобразования (яркость, контраст, цветовой сдвиг) применяются только к изображению;
- нормализация и `ToTensorV2` идут последними;
- в validation и test случайные аугментации не применяются никогда.

#### 1.5 Методика сравнения

Сравнение конфигураций корректно при совпадении: разбиения данных, разрешения входа, нормализации, числа эпох, оптимизатора и расписания learning rate, критерия выбора checkpoint (лучший val mIoU), seed. В каждой серии меняется ровно один фактор. Test используется один раз, после того как все решения приняты по validation.

### 2. Практическая часть

#### 2.1 Подготовка окружения

Версии `albumentations` и `segmentation-models-pytorch` закреплены по мажорной компоненте: во второй мажорной версии `albumentations` изменились имена и семантика ряда преобразований, и незакреплённая установка ломает воспроизводимость.

In [ ]:
# Версии закреплены по мажорной компоненте: незакреплённая установка ломает воспроизводимость.
%pip install -q "torch>=2.2,<3.0" "torchvision>=0.17,<1.0" "segmentation-models-pytorch>=0.4,<0.6" "albumentations>=1.4,<2.0" "torchmetrics>=1.4,<2.0" "timm>=1.0,<2.0" "pandas>=2.0,<3.0" "matplotlib>=3.8,<4.0"

In [ ]:
import json
import random
import time
from dataclasses import dataclass, asdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
import torchvision
from torchvision.datasets import OxfordIIITPet

import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
import torchmetrics
from torchmetrics.classification import MulticlassConfusionMatrix

SEED = 42
DATA_ROOT = Path("data")
OUTPUT_DIR = Path("outputs")
(OUTPUT_DIR / "checkpoints").mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("device:", DEVICE)
print("torch:", torch.__version__, "| torchvision:", torchvision.__version__)
print("smp:", smp.__version__, "| albumentations:", A.__version__,
      "| torchmetrics:", torchmetrics.__version__)

In [ ]:
def set_global_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


RUNS_PATH = OUTPUT_DIR / "runs.jsonl"
RUNS: list = []


def log_run(name: str, **fields) -> dict:
    """Записать конфигурацию и результат серии в журнал экспериментов."""
    record = {"name": name, "seed": SEED, "device": DEVICE.type, **fields}
    RUNS.append(record)
    with RUNS_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
    return record


def results_table(columns=None) -> pd.DataFrame:
    df = pd.DataFrame(RUNS)
    if columns:
        columns = [c for c in columns if c in df.columns]
        df = df[columns]
    return df


def count_trainable_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


set_global_seed()

#### 2.2 Данные: Oxford-IIIT Pet и trimap

Датасет загружается с `target_types="segmentation"`: целевой объект — PIL-изображение trimap со значениями `{1, 2, 3}`.

Соглашение, принятое в этой работе (фиксируется один раз и не меняется между сериями):

| trimap | класс | индекс |
|---|---|---|
| 1 | животное | 0 |
| 2 | фон | 1 |
| 3 | граница | 2 |

**Параметр `N_SUBSET`** ограничивает размер обучающей выборки. Сегментация дороже классификации: каждое изображение даёт $H \times W$ обучающих примеров, а декодер обучается с нуля. Подвыборка нужна, чтобы (1) уложить обучение baseline и серии из трёх конфигураций в 2 академических часа на одном бесплатном GPU; (2) обеспечить одинаковый бюджет данных у всех сравниваемых конфигураций. Абсолютные значения mIoU при этом будут ниже, чем на полном наборе, — ограничение фиксируется в выводе.

In [ ]:
N_SUBSET = 1200      # изображений в train
N_VAL = 400
N_TEST = 400
IMAGE_SIZE = 256
BATCH_SIZE = 8
NUM_CLASSES = 3
CLASS_NAMES = ["животное", "фон", "граница"]
IGNORE_INDEX = None   # альтернатива: 2 -- исключить границу из loss и метрик

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

trainval_raw = OxfordIIITPet(root=DATA_ROOT, split="trainval",
                             target_types="segmentation", download=True)
test_raw = OxfordIIITPet(root=DATA_ROOT, split="test",
                         target_types="segmentation", download=True)

rng = np.random.default_rng(SEED)
trainval_perm = rng.permutation(len(trainval_raw))
test_perm = rng.permutation(len(test_raw))

train_idx = [int(i) for i in trainval_perm[:N_SUBSET]]
val_idx = [int(i) for i in trainval_perm[N_SUBSET:N_SUBSET + N_VAL]]
test_idx = [int(i) for i in test_perm[:N_TEST]]

assert not (set(train_idx) & set(val_idx)), "train и validation пересекаются"
print({"train": len(train_idx), "val": len(val_idx), "test": len(test_idx)})


def trimap_to_mask(trimap: Image.Image) -> np.ndarray:
    """Перевести trimap {1,2,3} в маску классов {0,1,2} (int64)."""
    array = np.array(trimap, dtype=np.int64)
    return np.clip(array - 1, 0, NUM_CLASSES - 1)

In [ ]:
image, trimap = trainval_raw[train_idx[0]]
mask = trimap_to_mask(trimap)

print("размер изображения:", image.size, "| уникальные значения маски:", np.unique(mask))

figure, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(image.convert("RGB"))
axes[0].set_title("изображение")
axes[1].imshow(mask, vmin=0, vmax=NUM_CLASSES - 1)
axes[1].set_title("маска классов")
axes[2].imshow(image.convert("RGB"))
axes[2].imshow(mask, alpha=0.45, vmin=0, vmax=NUM_CLASSES - 1)
axes[2].set_title("наложение")
for axis in axes:
    axis.axis("off")
plt.tight_layout()
plt.show()

# TODO (задание 1.1): оцените дисбаланс классов -- посчитайте долю пикселей
# каждого класса на 100-200 изображениях train. Ответьте в отчёте: какое
# значение mIoU даст тривиальная модель, предсказывающая только самый частый класс?

#### 2.3 Аугментации

`train_transform` дан как рабочий baseline. Обратите внимание на порядок: сначала геометрия, затем фотометрия, затем нормализация, затем `ToTensorV2`. Изменение порядка (например, нормализация до фотометрических преобразований) меняет смысл аугментации.

Маска передаётся аргументом `mask=` и автоматически проходит те же геометрические преобразования; `Albumentations` использует для масок интерполяцию ближайшего соседа. Проверять это нужно всегда: появление в маске значений вне `{0, 1, 2}` означает ошибку в пайплайне.

In [ ]:
def build_train_transform(strength: str = "base") -> A.Compose:
    """Пайплайн аугментаций для train. strength -- метка серии сравнения."""
    if strength == "base":
        return A.Compose([
            A.Resize(IMAGE_SIZE, IMAGE_SIZE),
            A.HorizontalFlip(p=0.5),
            A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ToTensorV2(),
        ])
    # TODO (задание 2.1): реализуйте вариант strength == "strong".
    # Разрешено использовать A.Affine (сдвиг/масштаб/поворот),
    # A.RandomBrightnessContrast, A.HueSaturationValue, A.GaussNoise.
    # Требования: геометрические преобразования применяются к изображению и маске,
    # фотометрические -- только к изображению; порядок и итоговое разрешение
    # совпадают с базовым вариантом, иначе сравнение серий некорректно.
    raise NotImplementedError(f"неизвестная конфигурация аугментаций: {strength}")


eval_transform = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

#### 2.4 Dataset и DataLoader

Класс ниже готов. Отметьте два момента: маска приводится к `int64` (`CrossEntropyLoss` требует индексы классов типа `long`) и в eval-режиме применяется `eval_transform` без случайных операций.

In [ ]:
class PetSegmentationDataset(Dataset):
    """Oxford-IIIT Pet для семантической сегментации.

    Возвращает (image: FloatTensor[3,H,W], mask: LongTensor[H,W]).
    """

    def __init__(self, base_dataset, indices, transform: A.Compose):
        self.base = base_dataset
        self.indices = indices
        self.transform = transform

    def __len__(self) -> int:
        return len(self.indices)

    def __getitem__(self, position: int):
        image, trimap = self.base[self.indices[position]]
        image = np.array(image.convert("RGB"))
        mask = trimap_to_mask(trimap)
        augmented = self.transform(image=image, mask=mask)
        return augmented["image"], augmented["mask"].long()


def build_loaders(train_strength: str = "base", batch_size: int = BATCH_SIZE):
    train_dataset = PetSegmentationDataset(trainval_raw, train_idx,
                                           build_train_transform(train_strength))
    val_dataset = PetSegmentationDataset(trainval_raw, val_idx, eval_transform)
    test_dataset = PetSegmentationDataset(test_raw, test_idx, eval_transform)
    generator = torch.Generator().manual_seed(SEED)
    return (
        DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                   num_workers=2, drop_last=True, generator=generator),
        DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2),
        DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2),
    )


train_loader, val_loader, test_loader = build_loaders()
images, masks = next(iter(train_loader))
print("батч изображений:", images.shape, images.dtype)
print("батч масок:", masks.shape, masks.dtype, "| значения:", torch.unique(masks).tolist())
assert masks.max().item() < NUM_CLASSES, "в маске появились недопустимые метки"

#### 2.5 Модель

`smp.Unet` собирает U-Net с выбранным энкодером. Аргументы:

- `encoder_name` — имя энкодера (`resnet34`, `efficientnet-b0`, `mobilenet_v2`, ...);
- `encoder_weights="imagenet"` — предобученные веса энкодера; `None` — обучение с нуля;
- `in_channels=3`, `classes=NUM_CLASSES`;
- `activation=None` — **модель возвращает логиты**. Это обязательное условие: `CrossEntropyLoss` и `smp.losses.DiceLoss(from_logits=True)` ожидают именно логиты. Указание `activation="softmax"` приведёт к двойному применению softmax и молча испортит обучение.

Декодер инициализируется случайно и обучается с нуля, энкодер — дообучается. Отсюда типичный приём: пониженный learning rate для энкодера. В baseline для простоты используется единый learning rate; исследование раздельных lr — один из вариантов серии.

In [ ]:
ARCHITECTURES = {
    "unet": smp.Unet,
    "fpn": smp.FPN,
    "deeplabv3plus": smp.DeepLabV3Plus,
}


def build_segmentation_model(architecture: str = "unet",
                             encoder_name: str = "resnet34",
                             encoder_weights="imagenet") -> nn.Module:
    # encoder_weights=None -- обучение энкодера с нуля (вариант серии сравнения)
    """Создать сегментационную модель SMP, возвращающую логиты [B, C, H, W]."""
    if architecture not in ARCHITECTURES:
        raise ValueError(f"неизвестная архитектура: {architecture}")
    return ARCHITECTURES[architecture](
        encoder_name=encoder_name,
        encoder_weights=encoder_weights,
        in_channels=3,
        classes=NUM_CLASSES,
        activation=None,
    )


model = build_segmentation_model().to(DEVICE)
with torch.no_grad():
    probe = model(images[:2].to(DEVICE))
print("выход модели:", probe.shape, "| обучаемых параметров:", count_trainable_parameters(model))

# TODO (задание 3.1): сравните число параметров энкодера и декодера
# (model.encoder и model.decoder) и объясните, почему инициализация энкодера
# предобученными весами важнее, чем инициализация декодера.

#### 2.6 Функция потерь

Baseline — кросс-энтропия. Задание — собрать комбинированную функцию потерь и сравнить её с baseline в контролируемой серии.

`smp.losses.DiceLoss(mode="multiclass", from_logits=True)` принимает логиты формы `[B, C, H, W]` и целевые метки формы `[B, H, W]` типа `long` — те же аргументы, что и `nn.CrossEntropyLoss`, поэтому обе потери взаимозаменяемы в цикле обучения.

In [ ]:
def build_loss(kind: str = "ce", dice_weight: float = 0.5):
    """Вернуть callable(logits, target) -> scalar loss.

    kind:
        'ce'       -- nn.CrossEntropyLoss (baseline);
        'dice'     -- smp.losses.DiceLoss(mode='multiclass', from_logits=True);
        'ce_dice'  -- (1 - dice_weight) * CE + dice_weight * Dice.
    """
    ce_kwargs = {} if IGNORE_INDEX is None else {"ignore_index": IGNORE_INDEX}
    cross_entropy = nn.CrossEntropyLoss(**ce_kwargs)
    if kind == "ce":
        return cross_entropy
    if kind == "dice":
        return smp.losses.DiceLoss(mode="multiclass", from_logits=True,
                                   ignore_index=IGNORE_INDEX)
    if kind == "ce_dice":
        # TODO (задание 4.1): реализуйте комбинированную потерю.
        # Требования: обе компоненты считаются от одних и тех же логитов;
        # вес dice_weight фиксируется до начала серии и не подбирается по test.
        raise NotImplementedError
    raise ValueError(f"неизвестная функция потерь: {kind}")


# TODO (задание 4.2): объясните в отчёте, почему CE и Dice ведут себя по-разному
# при сильном дисбалансе классов и какой вклад каждая вносит на ранних эпохах.

#### 2.7 Обучение и валидация

Метрики считаются по накопленной матрице ошибок `MulticlassConfusionMatrix` из `torchmetrics`: из одной матрицы получаются и IoU, и Dice по классам. Для класса $c$:

$$\mathrm{IoU}_c = \frac{TP_c}{TP_c + FP_c + FN_c}, \qquad \mathrm{Dice}_c = \frac{2\,TP_c}{2\,TP_c + FP_c + FN_c}.$$

Метрики накапливаются по всем батчам, а не усредняются по батчам: усреднение побатчевых значений даёт смещённую оценку при разных долях классов в батчах.

In [ ]:
def per_class_scores(confusion: torch.Tensor) -> dict:
    """IoU и Dice по классам из матрицы ошибок [C, C] (строки -- истина)."""
    confusion = confusion.double()
    true_positive = confusion.diag()
    false_positive = confusion.sum(dim=0) - true_positive
    false_negative = confusion.sum(dim=1) - true_positive
    iou = true_positive / (true_positive + false_positive + false_negative).clamp(min=1e-9)
    dice = 2 * true_positive / (2 * true_positive + false_positive + false_negative).clamp(min=1e-9)
    return {
        "iou_per_class": iou.tolist(),
        "dice_per_class": dice.tolist(),
        "miou": float(iou.mean()),
        "mdice": float(dice.mean()),
    }


@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader) -> dict:
    """Оценить модель: mIoU, mDice, поклассовые значения и время инференса."""
    model.eval()
    confusion_metric = MulticlassConfusionMatrix(num_classes=NUM_CLASSES,
                                                 ignore_index=IGNORE_INDEX).to(DEVICE)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    started = time.perf_counter()
    for batch_images, batch_masks in loader:
        batch_images = batch_images.to(DEVICE)
        batch_masks = batch_masks.to(DEVICE)
        logits = model(batch_images)
        confusion_metric.update(logits.argmax(dim=1), batch_masks)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    scores = per_class_scores(confusion_metric.compute().cpu())
    scores["inference_time_s"] = time.perf_counter() - started
    return scores


def train_one_epoch(model: nn.Module, loader: DataLoader, loss_fn, optimizer) -> float:
    """Одна эпоха обучения; возвращает среднюю потерю на объект."""
    model.train()
    running_loss, seen = 0.0, 0
    for batch_images, batch_masks in loader:
        batch_images = batch_images.to(DEVICE)
        batch_masks = batch_masks.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        logits = model(batch_images)
        loss = loss_fn(logits, batch_masks)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * batch_images.size(0)
        seen += batch_images.size(0)
    return running_loss / max(seen, 1)

In [ ]:
@dataclass
class SegmentationConfig:
    name: str
    architecture: str = "unet"
    encoder_name: str = "resnet34"
    encoder_weights: str = "imagenet"     # None -- обучение энкодера с нуля
    augmentation: str = "base"
    loss: str = "ce"
    epochs: int = 5
    learning_rate: float = 3e-4
    batch_size: int = BATCH_SIZE


def run_experiment(config: SegmentationConfig) -> dict:
    """Выполнить одну воспроизводимую серию.

    Контракт:
        * фиксирует seed перед созданием модели и загрузчиков;
        * обучает config.epochs эпох, после каждой считает метрики на VALIDATION;
        * сохраняет checkpoint с лучшим val mIoU в outputs/checkpoints/<name>.pt;
        * загружает лучший checkpoint и оценивает его на TEST ровно один раз;
        * возвращает словарь с полями конфигурации (asdict(config)), val_miou,
          test_miou, test_mdice, iou_per_class, train_time_s, inference_time_s,
          trainable_params и историей значений по эпохам;
        * записывает результат через log_run.

    Запрещено: выбирать checkpoint или гиперпараметры по test.
    """
    raise NotImplementedError


# TODO (задание 5.1): реализуйте run_experiment.
# TODO (задание 5.2): запустите baseline и убедитесь, что val mIoU растёт
# и заметно превышает значение тривиальной модели из задания 1.1.

baseline_config = SegmentationConfig(name="baseline_unet_resnet34")
baseline_config

#### 2.8 Контролируемая серия

Выберите **один** фактор и проверьте его вклад. Допустимые варианты:

| Фактор | Baseline | Альтернатива |
|---|---|---|
| Инициализация энкодера | `encoder_weights="imagenet"` | `encoder_weights=None` |
| Декодер | `unet` | `fpn` или `deeplabv3plus` |
| Функция потерь | `ce` | `ce_dice` |
| Сила аугментаций | `base` | `strong` |
| Энкодер | `resnet34` | `efficientnet-b0` |

Правило: в серии меняется ровно один аргумент `SegmentationConfig`, остальные совпадают с baseline. Если одновременно изменить декодер и аугментации, прирост неинтерпретируем — это ошибка, отдельно отмеченная в рубрике оценивания.

Серия ограничена 2–3 конфигурациями сверх baseline, чтобы уложиться в бюджет практикума.

In [ ]:
research_question = ""   # TODO (задание 6.1): что именно проверяется
hypothesis = ""          # TODO (задание 6.2): ожидаемый результат и обоснование ДО запуска

# TODO (задание 6.3): опишите 2-3 конфигурации серии.
# Пример: единственное отличие от baseline -- encoder_weights.
series_configs = [
    # SegmentationConfig(name="no_pretrain", encoder_weights=None),
]

# TODO (задание 6.4): выполните серию через run_experiment и убедитесь,
# что все конфигурации обучались одинаковое число эпох на одном split.

#### 2.9 Сопоставимое сравнение

Сведите baseline и все конфигурации серии в одну таблицу. Строки таблицы сопоставимы только при совпадении разбиения, разрешения, нормализации, числа эпох и устройства; изменённый фактор указывается явно отдельным столбцом.

Обязательно приводится не только качество, но и стоимость: время обучения, время инференса и число обучаемых параметров. Сравнение архитектур без учёта времени — типичная ошибка.

In [ ]:
SUMMARY_COLUMNS = [
    "name", "architecture", "encoder_name", "encoder_weights", "augmentation",
    "loss", "epochs", "trainable_params", "train_time_s", "inference_time_s",
    "val_miou", "test_miou", "test_mdice",
]

summary = results_table(SUMMARY_COLUMNS)
summary

# TODO (задание 7.1): выведите таблицу и отдельно -- поклассовые IoU
# для baseline и лучшей конфигурации серии. Класс «граница» почти всегда
# даёт самый низкий IoU; объясните почему.
# TODO (задание 7.2): постройте кривые val mIoU по эпохам для всех конфигураций
# на одной фигуре.
# TODO (задание 7.3): постройте диаграмму «test mIoU vs время обучения»
# и укажите, какая конфигурация оправдывает свою стоимость.
# TODO (задание 7.4): проверьте согласованность ранжирования по val и по test.
# Если лучшая по validation конфигурация не лучшая на test -- это наблюдение,
# которое нужно объяснить, а не повод пересобрать выбор по test.

#### 2.10 Анализ ошибок

Среднее mIoU не показывает, где именно модель ошибается. Найдите изображения с худшим IoU и разберите их. Типичные источники ошибок на этом наборе: животное занимает малую часть кадра, несколько животных, слабый контраст с фоном, сложная шерсть по контуру, обрезанный объект на границе кадра.

In [ ]:
def visualize_prediction(dataset: PetSegmentationDataset, index: int,
                         model: nn.Module) -> None:
    """Показать изображение, истинную и предсказанную маски для одного объекта."""
    model.eval()
    image_tensor, mask_tensor = dataset[index]
    with torch.no_grad():
        prediction = model(image_tensor.unsqueeze(0).to(DEVICE)).argmax(dim=1)[0].cpu()
    denormalized = image_tensor.permute(1, 2, 0).numpy()
    denormalized = denormalized * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
    denormalized = np.clip(denormalized, 0, 1)
    figure, axes = plt.subplots(1, 3, figsize=(13, 4))
    axes[0].imshow(denormalized)
    axes[0].set_title("изображение")
    axes[1].imshow(mask_tensor, vmin=0, vmax=NUM_CLASSES - 1)
    axes[1].set_title("истинная маска")
    axes[2].imshow(prediction, vmin=0, vmax=NUM_CLASSES - 1)
    axes[2].set_title("предсказание")
    for axis in axes:
        axis.axis("off")
    plt.tight_layout()
    plt.show()


# TODO (задание 8.1): реализуйте функцию, вычисляющую IoU класса «животное»
# для каждого изображения test отдельно, и отберите 5 худших.
# TODO (задание 8.2): визуализируйте их через visualize_prediction и
# сформулируйте, что общего у этих случаев.
# TODO (задание 8.3): проверьте, исправляет ли лучшая конфигурация серии
# те же случаи или ошибается на них так же. Совпадающие ошибки указывают
# на ограничение данных, а не архитектуры.

### Отчёт

Отчётная часть включает:

1. **Условия эксперимента:** окружение и версии `torch`, `smp`, `albumentations`, `torchmetrics`; seed; `N_SUBSET`, `N_VAL`, `N_TEST`; разрешение `IMAGE_SIZE`; трактовка класса «граница» (третий класс или `ignore_index`); нормализация входа.
2. **Сводную таблицу** baseline и всех конфигураций серии: изменённый фактор, число обучаемых параметров, время обучения, время инференса, val mIoU, test mIoU, test mDice.
3. **Поклассовые IoU** для baseline и лучшей конфигурации.
4. **Не менее трёх визуализаций:** кривые val mIoU по эпохам, «качество vs время обучения», примеры предсказаний (включая худшие случаи).
5. **Гипотезу серии, сформулированную до запуска**, и её проверку: подтвердилась, опровергнута или результат неразличим в пределах наблюдаемого разброса.
6. **Анализ ошибок** по худшим изображениям с указанием предполагаемых причин.
7. **Выводы, отделённые от наблюдений**, с перечнем ограничений: подвыборка `N_SUBSET`, малое число эпох, один seed, одно разрешение, отсутствие оценки разброса по нескольким запускам.

Работа не засчитывается, если: конфигурации отличались более чем одним фактором без обоснования; сравнение проведено при разных разрешениях или аугментациях; checkpoint выбирался по test; приведён только лучший результат без журнала `runs.jsonl`.

### Контрольные вопросы

1. Чем семантическая сегментация отличается от instance-сегментации (ЛР4) и что изменится в разметке, если на изображении два животных?
2. Какую роль играют skip-соединения в U-Net и как изменится качество границ объекта без них?
3. Почему предобучение энкодера на ImageNet помогает в сегментации, хотя исходная задача — классификация?
4. Как связаны IoU и Dice? Какая из метрик всегда больше и почему обе нельзя приводить как «одно качество»?
5. Почему mIoU усредняется по классам, а не по пикселям?
6. Что даёт добавление Dice к кросс-энтропии и в каких данных эффект будет наибольшим?
7. Почему в `smp` модель создаётся с `activation=None` и что произойдёт при `activation="softmax"` вместе с `CrossEntropyLoss`?
8. Как `Albumentations` обеспечивает согласованность преобразований изображения и маски и почему маска не может интерполироваться линейно?
9. Почему в validation и test нельзя применять случайные аугментации?
10. Класс «граница» даёт наименьший IoU. Это дефект модели, свойство разметки или следствие постановки задачи?